# 성장 준비금 계좌 전략별 생존 검증

투자·인출 조건을 입력하고 아래 코드 셀을 실행하세요. 시작연도별 세후 잔액과 고갈 여부를 비교합니다.

In [ ]:
# @title 계좌 전략별 성장 준비금 조건을 입력하세요
최초_시작연도 = 2002  # @param {type:"integer", min:2000, step:1}
마지막_시작연도 = 2016  # @param {type:"integer", min:2000, step:1}
초기준비금_억원 = 4.0  # @param {type:"number", min:0.01, step:0.1}
월실수령액_만원 = 400  # @param {type:"number", min:0, step:10}
투자기간_년 = 10  # @param {type:"integer", min:1, step:1}
분할매수기간_개월 = 12  # @param {type:"integer", min:1, max:24, step:1}
미국계좌_우선금액_만원 = 4545  # @param {type:"number", min:0, step:1}
인출순서 = "미국계좌 잔여분 → 국내 일반계좌 → 연금저축 → ISA → 미국계좌 우선분"  # @param ["미국계좌 잔여분 → 국내 일반계좌 → 연금저축 → ISA → 미국계좌 우선분", "국내 일반계좌 → 미국계좌 → 연금저축 → ISA", "미국계좌 → 국내 일반계좌 → 연금저축 → ISA"]

"""QQQ 투자 시 계좌 전략별 세후 정기 인출 결과를 비교한다.

QQQ 수정주가는 Yahoo Finance, 원/달러 환율은 미국 연준 FRED의
DEXKOUS를 사용한다. 미투자 준비금은 이자가 없는 원화 현금으로 두고,
매월 말 세후 실수령액을 먼저 인출한 뒤 12개월 동안 분할매수한다.

절세계좌는 부부 두 사람의 ISA 누적 납입한도와 연금계좌 연간 한도를
활용한다. ISA와 연금계좌에서는 국내 상장 나스닥100 ETF가 QQQ와 같은
수익률을 낸다고 가정하며 상품 보수와 추적오차는 반영하지 않는다.
"""

from dataclasses import dataclass, replace
import math
import os
from pathlib import Path
import subprocess
import sys
from typing import Any, cast
from urllib.request import urlretrieve

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import font_manager
from matplotlib.axes import Axes
from matplotlib.patches import Patch
import pandas as pd
from IPython.display import Image, Markdown, display


def is_colab_runtime() -> bool:
    """현재 코드가 Google Colab에서 실행 중인지 확인한다."""
    return bool(os.environ.get("COLAB_RELEASE_TAG")) or Path("/content").exists()


IS_COLAB = is_colab_runtime()
OUTPUT_DIR = Path("/content/output") if IS_COLAB else Path("output")
WON_PER_EOK = 100_000_000
WON_PER_MANWON = 10_000
MONTHS_PER_YEAR = 12
TICKER = "QQQ"
FIRST_START_YEAR = int(최초_시작연도)
LAST_START_YEAR = int(마지막_시작연도)
INITIAL_RESERVE_KRW = float(초기준비금_억원) * WON_PER_EOK
MONTHLY_NET_WITHDRAWAL_KRW = float(월실수령액_만원) * WON_PER_MANWON
INVESTMENT_YEARS = int(투자기간_년)
DCA_MONTHS = int(분할매수기간_개월)
TOTAL_MONTHS = INVESTMENT_YEARS * MONTHS_PER_YEAR
WITHDRAWAL_ORDER_LABEL = str(인출순서)
INVESTOR_COUNT = 2
US_DEDUCTION_KRW = 2_500_000 * INVESTOR_COUNT
US_TAX_RATE = 0.22
GENERAL_TAX_RATE = 0.154
ISA_EXEMPTION_KRW = 2_000_000 * INVESTOR_COUNT
ISA_TAX_RATE = 0.099
PENSION_TAX_RATE = 0.165
ISA_MAX_KRW = 100_000_000 * INVESTOR_COUNT
PENSION_ANNUAL_LIMIT_KRW = 18_000_000 * INVESTOR_COUNT
GENERAL_MAX_KRW = 200_000_000 * INVESTOR_COUNT
US_CORE_PER_PERSON_KRW = float(미국계좌_우선금액_만원) * WON_PER_MANWON
US_CORE_KRW = US_CORE_PER_PERSON_KRW * INVESTOR_COUNT
STRATEGIES = ("세전 기준", "미국계좌", "절세계좌")
STRATEGY_COLORS = {
    "세전 기준": "#8D8B85",
    "미국계좌": "#2F7DD3",
    "절세계좌": "#1FAE7A",
}
BACKGROUND_COLOR = "#FFFFFF"
TEXT_COLOR = "#0B0B0B"
SECONDARY_TEXT_COLOR = "#64748B"
TICK_COLOR = "#777777"
GRID_COLOR = "#DEDCD6"
DEPLETION_COLOR = "#F06432"
REFERENCE_COLOR = "#F59E0B"
BRAND_SIZE = 13
TITLE_SIZE = 21
SUBTITLE_SIZE = 16
AXIS_TITLE_SIZE = 15
TICK_SIZE = 15
LEGEND_SIZE = 15
DATA_LABEL_SIZE = 14
FOOTNOTE_SIZE = 13

try:
    import yfinance as yf
except ImportError:
    if not IS_COLAB:
        raise
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])
    import yfinance as yf


@dataclass
class Account:
    """계좌의 현금, QQQ 수량과 세금 계산용 원금을 보관한다."""
    cash_krw: float = 0.0
    shares: float = 0.0
    basis_krw: float = 0.0


@dataclass
class Portfolio:
    """절세계좌 전략의 계좌와 남은 공제액을 보관한다."""
    us: Account
    general: Account
    pension: Account
    isa: Account
    us_excess_cash_krw: float = 0.0
    us_deduction_remaining_krw: float = US_DEDUCTION_KRW
    isa_exemption_remaining_krw: float = ISA_EXEMPTION_KRW
    taxes_paid_krw: float = 0.0


def validate_parameters() -> None:
    """입력값과 인출 순서를 검증한다."""
    if FIRST_START_YEAR < 2000 or FIRST_START_YEAR > LAST_START_YEAR:
        raise ValueError("시작연도 범위를 2000년 이후로 올바르게 입력하세요.")
    if INITIAL_RESERVE_KRW <= 0 or INVESTMENT_YEARS <= 0:
        raise ValueError("초기 준비금과 투자기간은 0보다 커야 합니다.")
    if MONTHLY_NET_WITHDRAWAL_KRW < 0:
        raise ValueError("월 실수령액은 0원 이상이어야 합니다.")
    if US_CORE_PER_PERSON_KRW < 0:
        raise ValueError("미국계좌 우선금액은 0원 이상이어야 합니다.")
    if not 1 <= DCA_MONTHS <= 24:
        raise ValueError("분할매수기간은 1~24개월이어야 합니다.")
    if WITHDRAWAL_ORDER_LABEL not in withdrawal_order_options():
        raise ValueError("지원하지 않는 인출 순서입니다.")


def withdrawal_order_options() -> dict[str, tuple[str, ...]]:
    """입력 문구별 계좌 인출 순서를 반환한다."""
    return {
        "미국계좌 잔여분 → 국내 일반계좌 → 연금저축 → ISA → 미국계좌 우선분": (
            "us_excess", "general", "pension", "isa", "us_core"
        ),
        "국내 일반계좌 → 미국계좌 → 연금계좌 → ISA": (
            "general", "us_core", "pension", "isa"
        ),
        "미국계좌 → 국내 일반계좌 → 연금계좌 → ISA": (
            "us_core", "general", "pension", "isa"
        ),
    }


def download_monthly_prices() -> pd.Series:
    """Yahoo Finance에서 QQQ 월말 수정주가를 내려받는다."""
    raw = yf.download(
        TICKER, start=f"{FIRST_START_YEAR - 1}-12-01",
        end=f"{LAST_START_YEAR + INVESTMENT_YEARS + 1}-01-10",
        auto_adjust=False, progress=False, actions=False,
    )
    if raw is None or raw.empty:
        raise RuntimeError("QQQ 가격 데이터를 내려받지 못했습니다.")
    prices = raw["Adj Close"]
    if isinstance(prices, pd.DataFrame):
        frame = cast(pd.DataFrame, prices)
        prices = frame[TICKER] if TICKER in frame.columns else frame.iloc[:, 0]
    prices = prices.dropna().astype(float)
    prices.index = pd.to_datetime(prices.index).tz_localize(None)
    return prices.resample("ME").last().rename(TICKER)


def download_monthly_usdkrw() -> pd.Series:
    """FRED에서 월말 원/달러 환율을 내려받는다."""
    start = f"{FIRST_START_YEAR - 1}-12-01"
    end = f"{LAST_START_YEAR + INVESTMENT_YEARS}-12-31"
    url = (
        "https://fred.stlouisfed.org/graph/fredgraph.csv"
        f"?id=DEXKOUS&cosd={start}&coed={end}"
    )
    raw = pd.read_csv(url)
    date_column = "observation_date" if "observation_date" in raw.columns else "DATE"
    dates = pd.to_datetime(raw[date_column])
    values = pd.to_numeric(raw["DEXKOUS"], errors="coerce")
    daily = pd.Series(values.to_numpy(), index=dates, name="usdkrw").dropna()
    return daily.resample("ME").last()


def load_market_data() -> pd.DataFrame:
    """QQQ 수정주가와 원/달러 환율을 결합한다."""
    return pd.concat([download_monthly_prices(), download_monthly_usdkrw()], axis=1)


def market_for_start(market: pd.DataFrame, start_year: int) -> pd.DataFrame:
    """한 시작연도의 10년 월말 데이터를 검증해 반환한다."""
    first_month = pd.Timestamp(start_year, 1, 31)
    trade_month = first_month - pd.offsets.MonthEnd(1)
    final_month = first_month + pd.offsets.MonthEnd(TOTAL_MONTHS - 1)
    required = market.loc[trade_month:final_month].copy()
    expected_count = TOTAL_MONTHS + 1
    if len(required) != expected_count or required.isna().any().any():
        raise RuntimeError(f"{start_year}년 시장 데이터 {expected_count}개월이 필요합니다.")
    return required


def account_value(account: Account, price_usd: float, fx_rate: float) -> float:
    """계좌의 원화 평가액을 계산한다."""
    return account.cash_krw + account.shares * price_usd * fx_rate


def buy_from_cash(account: Account, requested_krw: float, price_usd: float, fx_rate: float) -> float:
    """계좌 현금으로 QQQ를 매수하고 실제 매수액을 반환한다."""
    purchase_krw = min(max(requested_krw, 0.0), account.cash_krw)
    account.cash_krw -= purchase_krw
    account.shares += purchase_krw / (price_usd * fx_rate)
    account.basis_krw += purchase_krw
    return purchase_krw


def buy_sequentially_from_accounts(
    portfolio: Portfolio, requested_krw: float, price_usd: float, fx_rate: float
) -> float:
    """미국계좌 우선분부터 절세계좌·일반계좌·미국계좌 잔여분 순으로 매수한다."""
    remaining_krw = max(requested_krw, 0.0)
    purchased_krw = 0.0
    for name in ("us", "isa", "pension", "general"):
        if remaining_krw <= 0.5:
            break
        account = getattr(portfolio, name)
        amount_krw = buy_from_cash(
            account, remaining_krw, price_usd, fx_rate
        )
        purchased_krw += amount_krw
        remaining_krw -= amount_krw
    if remaining_krw > 0.5 and portfolio.us_excess_cash_krw > 0.0:
        amount_krw = min(remaining_krw, portfolio.us_excess_cash_krw)
        portfolio.us_excess_cash_krw -= amount_krw
        portfolio.us.shares += amount_krw / (price_usd * fx_rate)
        portfolio.us.basis_krw += amount_krw
        purchased_krw += amount_krw
    return purchased_krw


def sell_shares_for_net(
    account: Account, requested_net_krw: float, max_gross_krw: float,
    price_usd: float, fx_rate: float, tax_rate: float, exemption_krw: float,
) -> tuple[float, float, float]:
    """세금을 낸 뒤 목표 실수령액이 되도록 보유 수량을 매도한다."""
    share_value_krw = account.shares * price_usd * fx_rate
    gross_limit_krw = min(max(max_gross_krw, 0.0), share_value_krw)
    if requested_net_krw <= 0 or gross_limit_krw <= 0:
        return 0.0, 0.0, exemption_krw
    gain_ratio = (
        max(share_value_krw - account.basis_krw, 0.0) / share_value_krw
        if share_value_krw > 0 else 0.0
    )

    def net_from_gross(gross_krw: float) -> float:
        gain_krw = gross_krw * gain_ratio
        tax_krw = max(gain_krw - exemption_krw, 0.0) * tax_rate
        return gross_krw - tax_krw

    if gain_ratio <= 0 or requested_net_krw * gain_ratio <= exemption_krw:
        gross_needed_krw = requested_net_krw
    else:
        gross_needed_krw = (
            requested_net_krw - tax_rate * exemption_krw
        ) / (1 - tax_rate * gain_ratio)
    gross_sale_krw = min(max(gross_needed_krw, 0.0), gross_limit_krw)
    realized_gain_krw = gross_sale_krw * gain_ratio
    tax_krw = max(realized_gain_krw - exemption_krw, 0.0) * tax_rate
    net_krw = gross_sale_krw - tax_krw
    sold_fraction = gross_sale_krw / share_value_krw
    account.shares *= max(1 - sold_fraction, 0.0)
    account.basis_krw *= max(1 - sold_fraction, 0.0)
    exemption_krw = max(exemption_krw - realized_gain_krw, 0.0)
    return min(net_krw, net_from_gross(gross_limit_krw)), tax_krw, exemption_krw


def withdraw_net_from_account(
    account: Account, requested_net_krw: float, price_usd: float, fx_rate: float,
    tax_rate: float, exemption_krw: float = 0.0, floor_krw: float = 0.0,
) -> tuple[float, float, float]:
    """계좌 평가액 하한을 지키며 세후 실수령액을 인출한다."""
    available_gross_krw = max(account_value(account, price_usd, fx_rate) - floor_krw, 0.0)
    cash_withdrawal_krw = min(account.cash_krw, requested_net_krw, available_gross_krw)
    account.cash_krw -= cash_withdrawal_krw
    remaining_net_krw = requested_net_krw - cash_withdrawal_krw
    remaining_gross_krw = available_gross_krw - cash_withdrawal_krw
    sale_net_krw, tax_krw, exemption_krw = sell_shares_for_net(
        account, remaining_net_krw, remaining_gross_krw,
        price_usd, fx_rate, tax_rate, exemption_krw,
    )
    return cash_withdrawal_krw + sale_net_krw, tax_krw, exemption_krw


def withdraw_from_us_excess(
    portfolio: Portfolio, requested_net_krw: float, price_usd: float, fx_rate: float
) -> tuple[float, float]:
    """미국계좌 잔여 현금과 우선금액 초과 평가액에서 차례로 인출한다."""
    cash_krw = min(portfolio.us_excess_cash_krw, requested_net_krw)
    portfolio.us_excess_cash_krw -= cash_krw
    remaining_net_krw = requested_net_krw - cash_krw
    net_krw, tax_krw, exemption_krw = withdraw_net_from_account(
        portfolio.us, remaining_net_krw, price_usd, fx_rate, US_TAX_RATE,
        portfolio.us_deduction_remaining_krw, US_CORE_KRW,
    )
    portfolio.us_deduction_remaining_krw = exemption_krw
    return cash_krw + net_krw, tax_krw


def initialize_tax_portfolio() -> tuple[Portfolio, dict[str, float]]:
    """부부(2인) 기준 절세 우선순위로 초기 준비금을 배분한다."""
    remaining_krw = INITIAL_RESERVE_KRW
    allocations: dict[str, float] = {}
    for name, target_krw in (
        ("us", US_CORE_KRW),
        ("isa", ISA_MAX_KRW),
        ("pension", PENSION_ANNUAL_LIMIT_KRW),
    ):
        allocations[name] = min(remaining_krw, target_krw)
        remaining_krw -= allocations[name]
    allocations["general"] = min(remaining_krw, GENERAL_MAX_KRW)
    remaining_krw -= allocations["general"]
    allocations["us_excess"] = remaining_krw
    portfolio = Portfolio(
        us=Account(cash_krw=allocations["us"]),
        general=Account(cash_krw=allocations["general"]),
        pension=Account(cash_krw=allocations["pension"]),
        isa=Account(cash_krw=allocations["isa"]),
        us_excess_cash_krw=allocations["us_excess"],
    )
    return portfolio, allocations


def portfolio_value(portfolio: Portfolio, price_usd: float, fx_rate: float) -> float:
    """절세계좌 포트폴리오의 총평가액을 계산한다."""
    return portfolio.us_excess_cash_krw + sum(
        account_value(account, price_usd, fx_rate)
        for account in (portfolio.us, portfolio.general, portfolio.pension, portfolio.isa)
    )


def withdraw_from_tax_portfolio(
    portfolio: Portfolio, requested_net_krw: float, price_usd: float, fx_rate: float,
) -> tuple[float, float]:
    """입력한 계좌 순서에 따라 세후 실수령액을 마련한다."""
    remaining_net_krw = requested_net_krw
    total_tax_krw = 0.0
    for source in withdrawal_order_options()[WITHDRAWAL_ORDER_LABEL]:
        if remaining_net_krw <= 0.5:
            break
        if source == "us_excess":
            net_krw, tax_krw = withdraw_from_us_excess(
                portfolio, remaining_net_krw, price_usd, fx_rate
            )
        elif source == "us_core":
            net_krw, tax_krw, exemption_krw = withdraw_net_from_account(
                portfolio.us, remaining_net_krw, price_usd, fx_rate,
                US_TAX_RATE, portfolio.us_deduction_remaining_krw,
            )
            portfolio.us_deduction_remaining_krw = exemption_krw
        else:
            account = getattr(portfolio, source)
            tax_rate = {
                "general": GENERAL_TAX_RATE,
                "pension": PENSION_TAX_RATE,
                "isa": ISA_TAX_RATE,
            }[source]
            exemption_krw = portfolio.isa_exemption_remaining_krw if source == "isa" else 0.0
            net_krw, tax_krw, exemption_krw = withdraw_net_from_account(
                account, remaining_net_krw, price_usd, fx_rate, tax_rate, exemption_krw
            )
            if source == "isa":
                portfolio.isa_exemption_remaining_krw = exemption_krw
        remaining_net_krw -= net_krw
        total_tax_krw += tax_krw
    portfolio.taxes_paid_krw += total_tax_krw
    return requested_net_krw - max(remaining_net_krw, 0.0), total_tax_krw


def fund_annual_pension(
    portfolio: Portfolio, price_usd: float, fx_rate: float
) -> float:
    """미국계좌 잔여분과 일반계좌에서 연금저축 납입 재원을 마련한다."""
    remaining_krw = PENSION_ANNUAL_LIMIT_KRW
    net_krw, tax_krw = withdraw_from_us_excess(
        portfolio, remaining_krw, price_usd, fx_rate
    )
    portfolio.taxes_paid_krw += tax_krw
    portfolio.pension.cash_krw += net_krw
    remaining_krw -= net_krw
    for source, tax_rate in ((portfolio.general, GENERAL_TAX_RATE),):
        if remaining_krw <= 0.5:
            break
        net_krw, tax_krw, exemption_krw = withdraw_net_from_account(
            source, remaining_krw, price_usd, fx_rate, tax_rate
        )
        portfolio.taxes_paid_krw += tax_krw
        portfolio.pension.cash_krw += net_krw
        remaining_krw -= net_krw
    contributed_krw = PENSION_ANNUAL_LIMIT_KRW - max(remaining_krw, 0.0)
    buy_from_cash(portfolio.pension, contributed_krw, price_usd, fx_rate)
    return contributed_krw


def harvest_us_deduction(
    account: Account, remaining_deduction_krw: float, price_usd: float, fx_rate: float
) -> float:
    """연말에 남은 미국계좌 기본공제만큼 취득원가를 높인다."""
    invested_value_krw = account.shares * price_usd * fx_rate
    unrealized_gain_krw = max(invested_value_krw - account.basis_krw, 0.0)
    harvested_gain_krw = min(unrealized_gain_krw, remaining_deduction_krw)
    account.basis_krw += harvested_gain_krw
    return max(remaining_deduction_krw - harvested_gain_krw, 0.0)


def liquidation_value(
    account: Account, price_usd: float, fx_rate: float, tax_rate: float, exemption_krw: float
) -> float:
    """계좌를 전액 청산했을 때의 세후 금액을 계산한다."""
    invested_value_krw = account.shares * price_usd * fx_rate
    gain_krw = max(invested_value_krw - account.basis_krw, 0.0)
    tax_krw = max(gain_krw - exemption_krw, 0.0) * tax_rate
    return account.cash_krw + invested_value_krw - tax_krw


def simulate_single_account(
    market: pd.DataFrame, start_year: int, strategy: str
) -> tuple[dict[str, Any], pd.DataFrame]:
    """세전 기준 또는 미국계좌의 월별 인출 과정을 계산한다."""
    required = market_for_start(market, start_year)
    account = Account(cash_krw=INITIAL_RESERVE_KRW)
    monthly_purchase_krw = INITIAL_RESERVE_KRW / DCA_MONTHS
    deduction_remaining_krw = US_DEDUCTION_KRW
    total_withdrawn_krw = 0.0
    taxes_paid_krw = 0.0
    depletion_date = None
    rows = []
    for month_number, (date, values) in enumerate(required.iterrows()):
        price_usd = float(values[TICKER])
        fx_rate = float(values["usdkrw"])
        if month_number > 0 and date.month == 1:
            deduction_remaining_krw = US_DEDUCTION_KRW
        actual_withdrawal_krw = 0.0
        tax_krw = 0.0
        if month_number > 0:
            tax_rate = 0.0 if strategy == "세전 기준" else US_TAX_RATE
            exemption_krw = 0.0 if strategy == "세전 기준" else deduction_remaining_krw
            actual_withdrawal_krw, tax_krw, exemption_krw = withdraw_net_from_account(
                account, MONTHLY_NET_WITHDRAWAL_KRW, price_usd, fx_rate,
                tax_rate, exemption_krw,
            )
            if strategy == "미국계좌":
                deduction_remaining_krw = exemption_krw
            total_withdrawn_krw += actual_withdrawal_krw
            taxes_paid_krw += tax_krw
        if month_number < DCA_MONTHS:
            buy_from_cash(account, monthly_purchase_krw, price_usd, fx_rate)
        if strategy == "미국계좌" and date.month == 12:
            deduction_remaining_krw = harvest_us_deduction(
                account, deduction_remaining_krw, price_usd, fx_rate
            )
        ending_balance_krw = account_value(account, price_usd, fx_rate)
        if month_number > 0 and (
            actual_withdrawal_krw + 0.5 < MONTHLY_NET_WITHDRAWAL_KRW
            or ending_balance_krw <= 0.5
        ):
            depletion_date = date
        rows.append({
            "start_year": start_year, "strategy": strategy, "date": date,
            "actual_withdrawal_krw": actual_withdrawal_krw,
            "tax_paid_krw": tax_krw, "ending_balance_krw": ending_balance_krw,
        })
        if depletion_date is not None:
            break
    final_row = rows[-1]
    if depletion_date is None:
        final_price_usd = float(required.iloc[-1][TICKER])
        final_fx_rate = float(required.iloc[-1]["usdkrw"])
        final_tax_rate = 0.0 if strategy == "세전 기준" else US_TAX_RATE
        final_exemption_krw = 0.0 if strategy == "세전 기준" else deduction_remaining_krw
        final_balance_krw = liquidation_value(
            account, final_price_usd, final_fx_rate, final_tax_rate, final_exemption_krw
        )
    else:
        final_balance_krw = 0.0
    summary = {
        "start_year": start_year, "strategy": strategy,
        "depleted_within_period": depletion_date is not None,
        "depletion_month": depletion_date.strftime("%Y-%m") if depletion_date is not None else "-",
        "months_survived": max(len(rows) - 1, 0),
        "total_withdrawn_krw": total_withdrawn_krw,
        "taxes_paid_krw": taxes_paid_krw,
        "ending_balance_krw": final_balance_krw,
        "pre_liquidation_balance_krw": float(final_row["ending_balance_krw"]),
    }
    return summary, pd.DataFrame(rows)


def simulate_tax_strategy(
    market: pd.DataFrame, start_year: int
) -> tuple[dict[str, Any], pd.DataFrame]:
    """부부(2인) 절세계좌 전략의 월별 인출 과정을 계산한다."""
    required = market_for_start(market, start_year)
    portfolio, _ = initialize_tax_portfolio()
    monthly_purchase_krw = INITIAL_RESERVE_KRW / DCA_MONTHS
    total_withdrawn_krw = 0.0
    depletion_date = None
    rows = []
    for month_number, (date, values) in enumerate(required.iterrows()):
        price_usd = float(values[TICKER])
        fx_rate = float(values["usdkrw"])
        if month_number > 0 and date.month == 1:
            portfolio.us_deduction_remaining_krw = US_DEDUCTION_KRW
            if date.year > start_year:
                fund_annual_pension(portfolio, price_usd, fx_rate)
        actual_withdrawal_krw = 0.0
        tax_krw = 0.0
        if month_number > 0:
            actual_withdrawal_krw, tax_krw = withdraw_from_tax_portfolio(
                portfolio, MONTHLY_NET_WITHDRAWAL_KRW, price_usd, fx_rate
            )
            total_withdrawn_krw += actual_withdrawal_krw
        if month_number < DCA_MONTHS:
            buy_sequentially_from_accounts(
                portfolio, monthly_purchase_krw, price_usd, fx_rate
            )
        if date.month == 12:
            portfolio.us_deduction_remaining_krw = harvest_us_deduction(
                portfolio.us, portfolio.us_deduction_remaining_krw, price_usd, fx_rate
            )
        ending_balance_krw = portfolio_value(portfolio, price_usd, fx_rate)
        if month_number > 0 and (
            actual_withdrawal_krw + 0.5 < MONTHLY_NET_WITHDRAWAL_KRW
            or ending_balance_krw <= 0.5
        ):
            depletion_date = date
        rows.append({
            "start_year": start_year, "strategy": "절세계좌", "date": date,
            "actual_withdrawal_krw": actual_withdrawal_krw,
            "tax_paid_krw": tax_krw, "ending_balance_krw": ending_balance_krw,
        })
        if depletion_date is not None:
            break
    if depletion_date is None:
        final_price_usd = float(required.iloc[-1][TICKER])
        final_fx_rate = float(required.iloc[-1]["usdkrw"])
        final_balance_krw = portfolio.us_excess_cash_krw + sum((
            liquidation_value(
                portfolio.us, final_price_usd, final_fx_rate,
                US_TAX_RATE, portfolio.us_deduction_remaining_krw,
            ),
            liquidation_value(
                portfolio.general, final_price_usd, final_fx_rate, GENERAL_TAX_RATE, 0.0
            ),
            liquidation_value(
                portfolio.pension, final_price_usd, final_fx_rate, PENSION_TAX_RATE, 0.0
            ),
            liquidation_value(
                portfolio.isa, final_price_usd, final_fx_rate,
                ISA_TAX_RATE, portfolio.isa_exemption_remaining_krw,
            ),
        ))
    else:
        final_balance_krw = 0.0
    summary = {
        "start_year": start_year, "strategy": "절세계좌",
        "depleted_within_period": depletion_date is not None,
        "depletion_month": depletion_date.strftime("%Y-%m") if depletion_date is not None else "-",
        "months_survived": max(len(rows) - 1, 0),
        "total_withdrawn_krw": total_withdrawn_krw,
        "taxes_paid_krw": portfolio.taxes_paid_krw,
        "ending_balance_krw": final_balance_krw,
        "pre_liquidation_balance_krw": ending_balance_krw,
    }
    return summary, pd.DataFrame(rows)


def calculate_results(market: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """모든 시작연도와 계좌 전략의 요약·월별 결과를 계산한다."""
    summaries = []
    details = []
    for start_year in range(FIRST_START_YEAR, LAST_START_YEAR + 1):
        for strategy in ("세전 기준", "미국계좌"):
            summary, detail = simulate_single_account(market, start_year, strategy)
            summaries.append(summary)
            details.append(detail)
        summary, detail = simulate_tax_strategy(market, start_year)
        summaries.append(summary)
        details.append(detail)
    return pd.DataFrame(summaries), pd.concat(details, ignore_index=True)


def format_duration(months: int) -> str:
    """개월 수를 연·개월 문자열로 표시한다."""
    years, remaining_months = divmod(months, MONTHS_PER_YEAR)
    if remaining_months == 0:
        return f"{years}년"
    return f"{years}년 {remaining_months}개월" if years else f"{remaining_months}개월"


def format_korean_won(amount_krw: float) -> str:
    """원화 금액을 억원·만원 단위로 표시한다."""
    rounded_manwon = int(round(max(amount_krw, 0.0) / WON_PER_MANWON))
    eok, manwon = divmod(rounded_manwon, 10_000)
    if eok and manwon:
        return f"{eok:,}억 {manwon:,}만원"
    if eok:
        return f"{eok:,}억원"
    return f"{manwon:,}만원" if manwon else "0원"


FONT_CANDIDATES = ("Pretendard", "Apple SD Gothic Neo", "Noto Sans CJK KR", "Malgun Gothic")
COLAB_FONT_PATH = Path("/content/.fonts/Pretendard-Regular.otf")
COLAB_FONT_URL = ("https://raw.githubusercontent.com/orioncactus/pretendard/main/"
                  "packages/pretendard/dist/public/static/Pretendard-Regular.otf")


def configure_korean_font() -> None:
    """사용 가능한 한글 글꼴을 Matplotlib에 설정한다."""
    installed = {font.name for font in font_manager.fontManager.ttflist}
    font_name = next((name for name in FONT_CANDIDATES if name in installed), None)
    if font_name is None and IS_COLAB:
        COLAB_FONT_PATH.parent.mkdir(parents=True, exist_ok=True)
        if not COLAB_FONT_PATH.exists():
            urlretrieve(COLAB_FONT_URL, COLAB_FONT_PATH)
        font_manager.fontManager.addfont(COLAB_FONT_PATH)
        font_name = font_manager.FontProperties(fname=COLAB_FONT_PATH).get_name()
    if font_name:
        plt.rcParams["font.family"] = font_name
    plt.rcParams["axes.unicode_minus"] = False


def draw_chart_panel(ax: Axes, panel: pd.DataFrame, y_max: float) -> None:
    """한 시작연도 구간의 계좌 전략별 세후 잔액을 그린다."""
    years = sorted(panel["start_year"].unique())
    x_positions = list(range(len(years)))
    bar_width = 0.24
    offsets = (-bar_width, 0.0, bar_width)
    for strategy_index, (strategy, offset) in enumerate(zip(STRATEGIES, offsets)):
        strategy_rows = panel.loc[panel["strategy"] == strategy].set_index("start_year").loc[years]
        for x, (start_year, raw_row) in enumerate(strategy_rows.iterrows()):
            row = cast(Any, raw_row)
            depleted = bool(row.depleted_within_period)
            value = float(row.ending_balance_krw) / WON_PER_EOK
            height = 0.035 if depleted else value
            ax.bar(
                x + offset, height, width=bar_width * 0.88,
                color="#FFF4EF" if depleted else STRATEGY_COLORS[strategy],
                edgecolor=DEPLETION_COLOR if depleted else STRATEGY_COLORS[strategy],
                linewidth=1.4 if depleted else 0, hatch="///" if depleted else None, zorder=3,
            )
            label = (
                f"고갈\n{format_duration(int(row.months_survived))}"
                if depleted else f"{value:.2f}억"
            )
            year_peak = panel.loc[panel["start_year"] == start_year, "ending_balance_krw"].max() / WON_PER_EOK
            label_offset = 7 + strategy_index * 22 if year_peak < 0.2 else 7
            ax.annotate(
                label, xy=(x + offset, height), xytext=(0, label_offset),
                textcoords="offset points", ha="center", va="bottom",
                fontsize=DATA_LABEL_SIZE, fontweight="bold",
                color=DEPLETION_COLOR if depleted else TEXT_COLOR,
            )
    ax.axhline(
        INITIAL_RESERVE_KRW / WON_PER_EOK, color=REFERENCE_COLOR,
        linewidth=1.4, linestyle=(0, (4, 4)), alpha=0.55, zorder=1,
    )
    ax.set_ylim(0, y_max)
    ax.set_xticks(x_positions, [str(year) for year in years])
    ax.set_xlabel("투자 시작연도", fontsize=AXIS_TITLE_SIZE, color=TICK_COLOR, labelpad=10)
    ax.set_ylabel(f"{INVESTMENT_YEARS}년 후 세후 잔액(억원)", fontsize=AXIS_TITLE_SIZE, color=TICK_COLOR, labelpad=10)
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:g}"))
    ax.tick_params(axis="both", colors=TICK_COLOR, labelsize=TICK_SIZE, length=0)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.9, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.margins(x=0.06)


def save_comparison_chart(summary: pd.DataFrame, output_path: Path) -> None:
    """계좌 전략별 세후 최종 잔액을 패널 막대그래프로 저장한다."""
    configure_korean_font()
    years = sorted(summary["start_year"].unique())
    years_per_panel = 4
    panel_count = math.ceil(len(years) / years_per_panel)
    y_max = max(
        1.25 * INITIAL_RESERVE_KRW / WON_PER_EOK,
        float(summary["ending_balance_krw"].max()) / WON_PER_EOK * 1.18,
    )
    fig, axes = plt.subplots(
        panel_count, 1, figsize=(11.5, 4.2 * panel_count + 2.8),
        sharey=True, squeeze=False,
    )
    fig.patch.set_facecolor(BACKGROUND_COLOR)
    fig.text(0.075, 0.985, "대도시 연구실", ha="left", va="top", fontsize=BRAND_SIZE, color=SECONDARY_TEXT_COLOR)
    fig.suptitle(
        f"성장 준비금 {INVESTMENT_YEARS}년 생존 검증 | 계좌 전략별 결과",
        x=0.075, y=0.962, ha="left", fontsize=TITLE_SIZE, fontweight="bold", color=TEXT_COLOR,
    )
    fig.text(
        0.075, 0.930,
        f"초기 준비금 {format_korean_won(INITIAL_RESERVE_KRW).replace('억원', '억 원')} · "
        f"월 {format_korean_won(MONTHLY_NET_WITHDRAWAL_KRW).replace('만원', '만 원')} 인출 · "
        f"{DCA_MONTHS}개월 분할매수 · 부부(2인) 기준",
        ha="left", fontsize=SUBTITLE_SIZE, color=SECONDARY_TEXT_COLOR,
    )
    legend_items = [
        Patch(facecolor=STRATEGY_COLORS[strategy], label=strategy) for strategy in STRATEGIES
    ]
    fig.legend(
        handles=legend_items, loc="upper left", bbox_to_anchor=(0.075, 0.895),
        frameon=False, ncol=4, prop={"size": LEGEND_SIZE},
        handlelength=1.0, columnspacing=1.4,
    )
    for panel_index, ax in enumerate(axes[:, 0]):
        panel_years = years[panel_index * years_per_panel:(panel_index + 1) * years_per_panel]
        draw_chart_panel(ax, summary.loc[summary["start_year"].isin(panel_years)], y_max)
    fig.text(
        0.02, 0.008,
        "QQQ 배당·분할 및 원/달러 환율 반영\n"
        "※ 월 인출액은 세후 실수령액 기준이며, 매도 시 발생하는 세금은 계좌에서 추가 차감\n"
        "※ 계좌 배분·분할매수 순서: 미국계좌 우선분 → ISA → 연금저축 → 국내 일반계좌 → 미국계좌 잔여분\n"
        "※ 월 인출 순서: 미국계좌 잔여분 → 국내 일반계좌 → 연금저축 → ISA → 미국계좌 우선분\n"
        "※ ISA는 시작 시 부부(2인) 한도 2억 원을 확보해 유지하고, 연금저축은 매년 최대 3,600만 원 납입",
        fontsize=FOOTNOTE_SIZE, color=SECONDARY_TEXT_COLOR,
    )
    fig.tight_layout(rect=(0.035, 0.075, 0.97, 0.90), h_pad=2.0)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=200, bbox_inches="tight", facecolor=BACKGROUND_COLOR)
    plt.close(fig)


def save_results(
    summary: pd.DataFrame, detail: pd.DataFrame, output_dir: Path
) -> tuple[Path, Path, Path]:
    """요약·상세 CSV와 그래프 PNG를 저장한다."""
    output_dir.mkdir(parents=True, exist_ok=True)
    summary_path = output_dir / "qqq_account_strategy_summary.csv"
    detail_path = output_dir / "qqq_account_strategy_monthly_detail.csv"
    chart_path = output_dir / "qqq_account_strategy_ending_balance.png"
    summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
    detail.to_csv(detail_path, index=False, encoding="utf-8-sig")
    save_comparison_chart(summary, chart_path)
    return summary_path, detail_path, chart_path


def display_results(summary: pd.DataFrame, paths: tuple[Path, Path, Path]) -> None:
    """핵심 조건, 그래프와 저장 경로를 표시한다."""
    survival = (
        summary.assign(survived=~summary["depleted_within_period"])
        .groupby("strategy")["survived"].sum()
    )
    display(Markdown(
        "## 계좌 전략별 생존 시작연도 수\n\n"
        + "\n".join(
            f"- **{strategy}**: {int(survival[strategy])}/{LAST_START_YEAR - FIRST_START_YEAR + 1}"
            for strategy in STRATEGIES
        )
    ))
    display(Image(filename=str(paths[-1])))
    print("저장 완료:")
    for path in paths:
        print(path)


def main() -> None:
    """입력 검증부터 데이터 수집, 계산, 저장과 출력까지 실행한다."""
    validate_parameters()
    market = load_market_data()
    summary, detail = calculate_results(market)
    paths = save_results(summary, detail, OUTPUT_DIR)
    display_results(summary, paths)


main()
